# Prepare fingerprints — piece-wise extractor (VoxCeleb, one shard → one directory)

Runs the SNN architecture and piece-wise training scheme of
`training/tonotopic_plasticity_bound/train.py` (triplet-STDP + soft-refractory-gated
lateral inhibition, 128-dim/2-types-per-channel, 8 repeats per piece, 25ms-normalized
homeostasis) as a frozen feature extractor, and writes one fingerprint **movie** shard
(a small directory) per Kaggle input shard.

**Piece framing intentionally diverges from train.py for speed**: pieces are 200ms
with **no overlap** (`HOP_MS == WINDOW_MS == 200`, discarding a trailing partial
piece), vs. train.py's 200ms/100ms 50%-overlapping pieces (kept as-is there — it's
the visualization/tuning pipeline, not a speed-sensitive bulk job). With no overlap,
total simulated ms per wav is `T_clip * N_REPEATS` instead of ~double that under 50%
overlap, roughly halving compute; combined with running each piece's N_REPEATS
repeats as a single `net.run()` call (see the "In-run repeat-boundary snapshot
capture" comment in `_fingerprint_core_piecewise.build_network()`) instead of 8
separate calls, this notebook is ~6x faster end-to-end than the original per-repeat,
50%-overlap version, verified bit-for-bit equivalent on the per-run mechanics (see
`scratchpad/equivalence_sweep.py` from that change). The cost is coarser temporal
resolution in the frame sequence (consecutive frames no longer share audio context) —
an accepted tradeoff for this extractor, not something to also change in train.py.

Unlike the older `prepare_fingerprints.ipynb` (whole-clip, 8-epoch, 192-dim — a stale
architecture snapshot), this notebook produces one **frame per piece**, so each wav
yields a variable-length sequence of frames instead of a single static fingerprint.
See `training/be_ann_w_fingerprints/ann_backend_sequence.ipynb` for the matching
sequence-aware downstream ANN.

**Output is a DIRECTORY, not a single `.npz`** (see `_fingerprint_core_piecewise.
fingerprint_to_sample_piece` and the CSR stacking + save in the driver cell below) —
ragged, CSR-style (concatenated frames + a per-sample `n_pieces` count, offsets =
`cumsum([0]+n_pieces)`). The three big frame arrays are saved as PLAIN `.npy` files,
not bundled into a `.npz`: an `.npz` is a zip archive, and numpy does **not** actually
lazy-load zip members even with `mmap_mode='r'` (verified — it silently reads the
whole array into RAM). At full-corpus scale (~95GB total) that distinction is the
difference between the downstream ANN notebook fitting in Kaggle RAM or OOMing.

| file | shape | dtype | meaning |
|---|---|---|---|
| `weights_frames.npy` | `(TOTAL_FRAMES,2,23,128)` | f16 | one STDP weight type-image per piece, silent cells zeroed |
| `input_activity_frames.npy` | `(TOTAL_FRAMES,128)` | f16 | per-piece input firing rate ∈ [0,1] |
| `hidden_activity_frames.npy` | `(TOTAL_FRAMES,128)` | f16 | per-piece hidden firing rate ∈ [0,1] |
| `meta.npz` (tiny, always loaded eagerly) | — | — | `n_pieces (N,) i32` (offsets = `cumsum([0]+n_pieces)`), `person_ids`/`record_ids`/`labels (N,) str`, provenance scalars |

Each piece's weights are the **average of the last 4 of its 8 repeats**
(`SNAPSHOT_FROM_REPEAT=4`), with input/hidden activity summed over that same window
— mirrors the old whole-clip pipeline's `SNAPSHOT_FROM_EPOCH=4` pattern, applied
per-piece instead of per-epoch. Only in→hid (excitatory) weights are captured;
hid→hid (inhibitory) is simulated (it's part of the network dynamics) but not saved.

**Workflow** — you have many shards and run this notebook once per shard:
1. Edit the **config cell**: point `INPUT_ROOT` at the shard folder, set `OUTPUT_NAME`
   (e.g. `vox2_200person_shard01_fingerprints_pw`, or `vox1_...` for the test set —
   this is now a DIRECTORY name, not a `.npz` filename).
2. Run all cells. The shard directory lands in `/kaggle/working/{OUTPUT_NAME}/`; the
   last cell prints a `zip -0` command to package it (uncompressed, so the `.npy`
   files stay mmap-able after Kaggle unzips the dataset). Upload the zips together as
   one Kaggle dataset for `ann_backend_sequence.ipynb`.

**Runtime note**: this scheme runs one `net.run()` per piece (~24 pieces per ~5s wav
at 200ms/no-overlap), each covering all 8 repeats in a single call. Pilot a handful of
wavs at `WORKERS=1` before committing to a full run — see the config cell.

The `%%writefile` cells recreate the extractor + audio encoder as real modules in the
working dir so that multiprocessing `spawn` workers can import them (a notebook has no
importable `__main__`). Requires `gammatone` (pip below) and `ffmpeg` (preinstalled on
Kaggle) to decode `.m4a`.


In [ ]:
# ── Setup: gammatone filterbank (Kaggle already ships ffmpeg, librosa, soundfile, brian2)
!pip -q install gammatone 2>/dev/null || pip -q install git+https://github.com/detly/gammatone.git
!pip -q install brian2
import shutil
assert shutil.which("ffmpeg"), "ffmpeg not on PATH — needed to decode .m4a"
print("ffmpeg:", shutil.which("ffmpeg"))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG  —  the only cell you edit between shards
# ══════════════════════════════════════════════════════════════════════════════
import os

# Kaggle input shard folder: {INPUT_ROOT}/{person_id}/{session_id}/{wav}.m4a
INPUT_ROOT  = "/kaggle/input/datasets/qphulong/vox2-voices-200person-shard01"

# Output shard DIRECTORY name (created under WORK_DIR, holding weights_frames.npy /
# input_activity_frames.npy / hidden_activity_frames.npy / meta.npz -- see intro
# markdown for why plain .npy, not one bundled .npz). Use a per-shard name so many
# shards can be uploaded together later:
#   dev  set : vox2_200person_shard01_fingerprints_pw/, ..._shard02_.../, ...
#   test set : vox1_200person_shard01_fingerprints_pw/, ...
# ("_pw" = piece-wise, to keep these visually distinct from the old whole-clip shards)
OUTPUT_NAME = "vox2_200person_shard01_fingerprints_pw"

WORK_DIR    = "/kaggle/working"
PER_SESSION = 1                     # at most N wavs per recording session (first N, sorted)
WORKERS     = os.cpu_count() or 2   # parallel Brian2 worker processes (Kaggle ~4)
AUDIO_EXTS  = (".m4a", ".wav")      # VoxCeleb2 = m4a; VoxCeleb1 (repo) may be wav

# Piece-wise extraction runs ~48x more (shorter) net.run() calls per wav than the
# old whole-clip scheme — set WORKERS=1 and point INPUT_ROOT at a tiny 5-10 wav
# folder first to get a real seconds/wav estimate before a full shard run.

print(f"INPUT_ROOT  = {INPUT_ROOT}")
print(f"OUTPUT_NAME = {OUTPUT_NAME}")
print(f"PER_SESSION = {PER_SESSION}  |  WORKERS = {WORKERS}")


### Write the extractor modules to the working dir (imported by spawn workers)

In [ ]:
%%writefile audio_utils.py
import shutil
import subprocess

import numpy as np
import librosa
import soundfile as sf
from gammatone.filters import centre_freqs, make_erb_filters, erb_filterbank


def _ffprobe_sample_rate(path):
    """Native sample rate of `path` via ffprobe, or None if it can't be determined."""
    ffprobe = shutil.which("ffprobe")
    if ffprobe is None:
        return None
    try:
        out = subprocess.run(
            [ffprobe, "-v", "error", "-select_streams", "a:0",
             "-show_entries", "stream=sample_rate",
             "-of", "default=noprint_wrappers=1:nokey=1", path],
            check=True, capture_output=True, text=True,
        ).stdout.strip().splitlines()
        return int(out[0]) if out else None
    except (subprocess.CalledProcessError, ValueError):
        return None


def _ffmpeg_decode(path, sr):
    """Decode any ffmpeg-readable container (m4a/aac/mp3/...) to a mono float32
    waveform. If `sr` is None the native rate is probed and preserved; otherwise
    ffmpeg's resampler outputs directly at `sr`."""
    ffmpeg = shutil.which("ffmpeg")
    if ffmpeg is None:
        raise RuntimeError(
            f"Cannot decode {path!r}: libsndfile failed and ffmpeg is not on PATH. "
            "Install ffmpeg to read m4a/aac audio."
        )
    target_sr = sr if sr is not None else (_ffprobe_sample_rate(path) or 16000)
    proc = subprocess.run(
        [ffmpeg, "-nostdin", "-loglevel", "error", "-i", path,
         "-ac", "1", "-ar", str(target_sr),
         "-f", "f32le", "-acodec", "pcm_f32le", "-"],
        capture_output=True,
    )
    if proc.returncode != 0:
        raise RuntimeError(
            f"ffmpeg failed to decode {path!r}: "
            f"{proc.stderr.decode('utf-8', 'ignore').strip()}"
        )
    y = np.frombuffer(proc.stdout, dtype="<f4").astype(np.float32)
    return y, target_sr


def load_audio(path, sr=16000):
    """Load `path` to a mono float32 waveform at `sr` Hz (native rate if `sr` is None).

    Format-robust replacement for ``librosa.load``: wav/flac/ogg are decoded by
    libsndfile (soundfile); m4a/aac and any container libsndfile can't open are
    decoded via ffmpeg. This avoids librosa's audioread m4a fallback, which is
    deprecated and slated for removal in librosa 1.0 (and emits a warning per file).
    """
    try:
        y, sr_native = sf.read(path, dtype="float32", always_2d=False)
    except Exception:
        return _ffmpeg_decode(path, sr)
    if y.ndim > 1:                       # multi-channel -> mono (match librosa default)
        y = y.mean(axis=1)
    if sr is not None and sr_native != sr:
        y = librosa.resample(y, orig_sr=sr_native, target_sr=sr)
    return np.ascontiguousarray(y, dtype=np.float32), (sr if sr is not None else sr_native)


def load_mel_spectrogram(
    wav_path: str,
    n_mels: int = 256,
    fmax: int | None = 8000,
    target_frames_per_second: int = 1000,
    normalize: bool = True,
):
    audio, sr = load_audio(wav_path, sr=None)

    hop_length = int(sr / target_frames_per_second)

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=n_mels,
        fmax=fmax,
        hop_length=hop_length
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)

    if normalize:
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)

    return mel_db, sr

def auditory_frontend(
    audio_path,
    sr=16000,
    num_filters=100,
    f_min=50,
    alpha=1.0,
    norm_percentile=99.0,
    clip_val=1.0,
    per_channel=False,
    eps=1e-6,
):
    """
    Encode an audio waveform into auditory-inspired spike features

    This function implements a biologically inspired auditory pipeline:
    waveform → gammatone filterbank → upstream percentile normalization →
    inner hair cell compression → onset detection → phase signal.

    Normalization is applied **once, upstream** to the (signed) filterbank output,
    before any nonlinearity. Because `log1p(alpha * x)` is not scale-invariant, the
    signal must be brought to a known scale *before* the log so compression is
    consistent across utterances. E, dE and phase are then all derived from the same
    normalized signal — their relative balance is therefore set only by the downstream
    gains, not by independent per-feature normalizations.

    Parameters
    ----------
    audio_path : str
        Path to the input audio file.

    sr : int, default=16000
        Target sampling rate for loading audio.

    num_filters : int, default=100
        Number of ERB-spaced gammatone filters (frequency channels).

    f_min : float, default=50
        Minimum center frequency (Hz) of the filterbank.

    alpha : float, default=1.0
        Compression strength for inner hair cell log compression:
        E = log1p(alpha * |signal_norm|).

    norm_percentile : float, default=99.0
        Percentile of |filterbank output| used as the normalization scale. Robust to
        the loudest transients (top 1% at 99) compared to a plain max.

    clip_val : float, default=1.0
        After dividing by the percentile scale, the normalized signal is clipped to
        [-clip_val, clip_val]. This bounds the input to the log and saturates the
        loudest excursions.

    per_channel : bool, default=False
        If False (default), one global percentile scalar is computed over the whole
        (n_channels, T) magnitude array — this preserves cross-channel relative energy
        (formant/timbre structure useful for speaker discrimination). If True, the
        percentile is computed per channel, equalizing quiet and loud bands.

    eps : float, default=1e-6
        Small constant to avoid division by zero.

    Returns
    -------
    dict
        Dictionary containing encoded auditory representations:

        - "E" : np.ndarray (n_channels, T)
            Log-compressed cochlear energy (IHC output), full-wave rectified.

        - "dE" : np.ndarray (n_channels, T)
            Onset detection signal (half-wave rectified temporal derivative of E).

        - "phase" : np.ndarray (n_channels, T)
            Negative half-wave of the normalized filterbank output. Complementary in
            polarity to E's full-wave energy, so it is not redundant with E.

        - "cf" : np.ndarray (n_channels,)
            Center frequencies of filterbank channels (low → high)

        - "sr" : int
            Sampling rate of processed audio

    Notes
    -----
    Processing pipeline:

    1. Audio loading
    2. ERB-spaced gammatone filterbank
    3. Upstream percentile normalization + clip (on the signed signal)
    4. Inner hair cell log compression (full-wave): E = log1p(alpha * |sig_norm|)
    5. Onset detection via positive temporal derivative of E
    6. Phase signal: negative half-wave of sig_norm

    All channel outputs are ordered from **low → high frequency**.
    """

    # ==============================
    # 1. Load audio
    # ==============================
    signal, sr = load_audio(audio_path, sr=sr)

    # ==============================
    # 2. Gammatone filterbank
    # ==============================
    cf = centre_freqs(sr, num_filters, f_min)
    erb_filters = make_erb_filters(sr, cf)

    filtered_signals = erb_filterbank(signal, erb_filters)

    # reorder HIGH→LOW → LOW→HIGH
    cf = cf[::-1]
    filtered_signals = filtered_signals[::-1]

    signals = filtered_signals
    n_channels, T = signals.shape

    # ==============================
    # 3. Upstream percentile normalization (on the signed signal, before any
    #    nonlinearity). One scale derived from |signals|, then clip. This keeps
    #    the log compression in a consistent regime across utterances and puts
    #    E / dE / phase on a single shared reference frame.
    # ==============================
    if per_channel:
        scale = np.percentile(np.abs(signals), norm_percentile, axis=1, keepdims=True)
    else:
        scale = np.percentile(np.abs(signals), norm_percentile)
    sig_n = np.clip(signals / (scale + eps), -clip_val, clip_val)

    # ==============================
    # 4. Inner Hair Cell Compression (full-wave)
    # ==============================
    E = np.log1p(alpha * np.abs(sig_n))

    # ==============================
    # 5. Onset detection (positive temporal derivative of E)
    # ==============================
    dE = np.diff(E, axis=1, prepend=E[:, :1])
    dE[dE < 0] = 0

    # ==============================
    # 6. Phase signal: negative half-wave of the normalized signal.
    #    Complementary in polarity to E's full-wave energy → not redundant with E.
    # ==============================
    phase_signal = np.maximum(-sig_n, 0)

    return {
        "E": E,
        "dE": dE,
        "phase": phase_signal,
        "cf": cf,
        "sr": sr,
    }


In [ ]:
%%writefile spike_encoding.py
import numpy as np
from audio_utils import auditory_frontend

def compute_spike_input_current(
    audio_path,
    sustained_per_band=5,
    onset_per_band=2,
    phase_per_band=2,
    scale=1,
    sust_gain=1.0,
    onset_gain=2.0,
    phase_gain=1.0,
    sust_spread_min=0.6,
    sust_spread_max=1.4,
    audio_sample_rate=16000,
    simulation_sample_rate=1000,
    num_filters=100,
    norm_percentile=99.0,
    clip_val=1.0,
    per_channel=False,
):
    """
    Convert an audio file into a downsampled input current matrix for a spiking neural network

    This function takes auditory features produced by `auditory_frontend()` and expands
    them into multiple neuron types per cochlear frequency band. Each neuron type
    represents different auditory response characteristics inspired by biological
    auditory nerve fibers.

    Pipeline
    --------
    1. Audio → auditory feature extraction via `auditory_frontend()`
    2. Obtain three feature maps:
        - E     : sustained energy (IHC compressed output)
        - dE    : onset energy (positive temporal derivative)
        - phase : rectified gammatone signal
    3. Generate multiple neurons per frequency band:
        - sustained neurons (energy response, spread across gain multipliers)
        - onset neurons (transient response)
        - phase neurons (phase locking)
    4. Apply gain scaling and small Gaussian noise.
    5. Downsample from `audio_sample_rate` to `simulation_sample_rate` by
       block-averaging across the decimation factor.
    6. Return a time-varying current matrix suitable for driving LIF neurons.

    Parameters
    ----------
    audio_path : str
        Path to the input audio (.wav) file.

    sustained_per_band : int, default=5
        Number of neurons per frequency band that encode sustained energy (E).

    onset_per_band : int, default=2
        Number of neurons per band that encode onset activity (dE).

    phase_per_band : int, default=2
        Number of neurons per band that encode phase-locking signals.

    scale : float, default=1
        Global gain multiplier applied to all input currents.

    sust_gain : float, default=1.3
        Gain factor for sustained-energy neurons.

    onset_gain : float, default=2.0
        Gain factor for onset neurons.

    phase_gain : float, default=1.0
        Gain factor for phase-locking neurons.

    sust_spread_min : float, default=0.6
        Minimum multiplicative factor applied across sustained neurons within a band.

    sust_spread_max : float, default=1.4
        Maximum multiplicative factor applied across sustained neurons within a band.

    audio_sample_rate : int, default=16000
        Sampling rate (Hz) of the raw audio and the auditory feature maps produced
        by `auditory_frontend()`.

    simulation_sample_rate : int, default=1000
        Target sampling rate (Hz) for the Brian2 simulation (i.e. 1 / defaultclock.dt).
        The current matrix is downsampled from `audio_sample_rate` to this rate by
        block-averaging. Must evenly divide `audio_sample_rate`.

    norm_percentile : float, default=99.0
        Percentile used by `auditory_frontend` to normalize the filterbank output
        before the log nonlinearity.

    clip_val : float, default=1.0
        Clip bound applied to the normalized filterbank signal in `auditory_frontend`.

    per_channel : bool, default=False
        If True, `auditory_frontend` normalizes per channel instead of globally.

    Returns
    -------
    I_sim : np.ndarray, shape (N_in, T_sim)
        Simulation-ready input current matrix, where
        T_sim = T // decimation_factor.

    T_sim : int
        Number of time steps after downsampling, corresponding to the
        total simulation duration in Brian2 timesteps.
    """

    feats = auditory_frontend(
        audio_path,
        num_filters=num_filters,
        norm_percentile=norm_percentile,
        clip_val=clip_val,
        per_channel=per_channel,
    )

    E = feats["E"]
    dE = feats["dE"]
    phase = feats["phase"]

    n_channels, T = E.shape

    g_sust = sust_gain
    g_onset = onset_gain
    g_phase = phase_gain

    neurons_per_band = sustained_per_band + onset_per_band + phase_per_band
    N_in = n_channels * neurons_per_band

    I = np.zeros((N_in, T), dtype=np.float32)

    idx = 0
    for ch in range(n_channels):

        spread = np.linspace(sust_spread_min, sust_spread_max, sustained_per_band)
        for mult in spread:
            I[idx] = g_sust * mult * scale * E[ch]
            idx += 1

        for _ in range(onset_per_band):
            I[idx] = g_onset * scale * dE[ch]
            idx += 1

        for _ in range(phase_per_band):
            I[idx] = g_phase * scale * phase[ch]
            idx += 1

    I += 0.01 * np.random.randn(*I.shape).astype(np.float32)
    assert audio_sample_rate % simulation_sample_rate == 0, (
        f"audio_sample_rate ({audio_sample_rate}) must be divisible by "
        f"simulation_sample_rate ({simulation_sample_rate})"
    )
    decimation_factor = audio_sample_rate // simulation_sample_rate
    # Trim to nearest multiple so reshape never fails
    T_trim = (T // decimation_factor) * decimation_factor
    I_sim  = I[:, :T_trim].reshape(N_in, -1, decimation_factor).mean(axis=2)
    T_sim  = I_sim.shape[1]

    return I_sim, T_sim


In [ ]:
%%writefile _fingerprint_core_piecewise.py
"""
_fingerprint_core_piecewise.py
===============================
Piece-wise fingerprint extractor core for be_ann_w_fingerprints.

The Brian2 network and every hyperparameter below are copied VERBATIM from
training/tonotopic_plasticity_bound/train.py (N_IN=N_H=128, 64 tonotopic channels x
2 neuron types [sustained, onset — phase disabled], triplet-STDP input->hidden,
Jaccard-scaled inhibitory hidden->hidden STDP gated by a soft-refractory trace,
homeostatic L1 column normalisation every 25ms). Unlike the older, stale
`_fingerprint_core_2.py` (192-dim/3-types-per-channel, whole-clip 8-epoch training),
this module reproduces train.py's PIECE-WISE training scheme: one clip is sliced
into WINDOW_MS pieces (HOP_MS == WINDOW_MS, no overlap -- see the "Piece framing"
comment above WINDOW_MS/HOP_MS below for why), each piece trained independently
from the SAME shared init weights for N_REPEATS repeated exposures (no restore
between repeats -- state carries over continuously within a piece).

One fingerprint FRAME is produced per piece, so one wav yields a variable-length
SEQUENCE of frames (a "fingerprint movie") instead of one static fingerprint:
  train_fingerprint_piecewise() : per piece, average the final in->hid weight
                                   matrix over the last SNAPSHOT_FROM_REPEAT..N_REPEATS-1
                                   repeats, and sum input/hidden spike counts over
                                   that same window (see fingerprint_to_sample_piece).
  fingerprint_to_sample_piece() : (128,128) mean weights + summed counts -> one
                                   frame's ANN-ready output:
                                     weights         (2,23,128) float16
                                     input_activity  (128,)     float16
                                     hidden_activity (128,)     float16

Only in->hid (excitatory) weights are captured per frame -- hid->hid (inhibitory)
is out of scope for this extractor (still simulated, since it's part of the
network dynamics, just not saved).

Worker helpers (_init_worker / process_one) live here so multiprocessing `spawn`
workers can import them from a real module (a notebook has no importable __main__).

Import this module BEFORE numpy in driver code so the BLAS-thread pinning below
takes effect (prevents oversubscription when many worker processes run in parallel).
"""

import os

# ── Pin BLAS threads BEFORE numpy is imported anywhere in the process ──────────
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")

import sys
from types import SimpleNamespace

import numpy as np

# Make the sibling encoder modules importable whether imported by path or re-imported
# by a spawned worker (cwd is not guaranteed to be on sys.path in a spawn child).
_HERE = os.path.dirname(os.path.abspath(__file__))
if _HERE not in sys.path:
    sys.path.insert(0, _HERE)
from spike_encoding import compute_spike_input_current

from brian2 import (
    NeuronGroup, Synapses, SpikeMonitor, TimedArray, Network, network_operation,
    defaultclock, prefs, BrianLogger, ms, second,
)

# ── Brian2 codegen / logging prefs ─────────────────────────────────────────────
prefs.codegen.target = 'cython'                          # JIT to C (gcc present)
prefs.codegen.runtime.cython.multiprocess_safe = True    # safe parallel build cache
BrianLogger.suppress_name('method_choice')
BrianLogger.suppress_name('unused_brian_object')         # warm-compile net is discarded
prefs.logging.console_log_level = 'ERROR'                # quiet across many workers

# ═══════════════════════════════════════════════════════════════════════════════
# Hyperparameters  (verbatim from tonotopic_plasticity_bound/train.py, N=128)
# ═══════════════════════════════════════════════════════════════════════════════

N_IN = 128   # 64 channels x 2 neurons/channel (sustained + onset, no phase)
N_H  = 128

DT_SIM = 1 * ms

# -- Piece framing --
# HOP_MS == WINDOW_MS: no overlap between consecutive pieces (was HOP_MS=100,
# 50% overlap). With no overlap, total simulated ms per wav collapses to
# T_clip * N_REPEATS regardless of WINDOW_MS (piece count * (WINDOW_MS *
# N_REPEATS) = (T_clip/WINDOW_MS) * WINDOW_MS * N_REPEATS), vs. ~2x that under
# 50% overlap -- roughly halves compute on top of the single-run-per-piece
# change in train_fingerprint_piecewise(). WINDOW_MS is kept at 200 (unchanged)
# since every other time constant (tau_h, tau_vth, tau_a, ...) was tuned
# assuming it, and a larger window buys no extra speed once there's no overlap.
# A trailing partial piece (< WINDOW_MS of audio) is discarded, same as before.
WINDOW_MS = 200   # piece length
HOP_MS    = 200   # slide between consecutive pieces (== WINDOW_MS: no overlap)
N_REPEATS = 8     # repeated exposures per piece

# -- Fingerprint collection: average the LAST N repeats of each piece (mirrors the
#    old whole-clip pipeline's SNAPSHOT_FROM_EPOCH=4 pattern, applied per-piece) --
SNAPSHOT_FROM_REPEAT = 4   # 0-indexed: average repeats 4..N_REPEATS-1 (last half of 8)

# -- Input layer (adaptive LIF) --
tau_m       = 40 * ms
tau_a       = 40 * ms
tau_current = 1 * ms
beta        = 3.5
v_th_in     = 1.0

# -- Hidden layer (adaptive-threshold LIF) --
tau_h    = 150 * ms
tau_vth  = 60 * ms
vth_rest = 0.6
vth_init = 0.6
vth_jump = 1.0

# -- Soft refractory (hidden only; replaces hard refractory) --
tau_r = 10 * ms

# -- Membrane noise (hidden only) --
sigma_noise = 0.03 * second**(-0.5)

# -- STDP (excitatory pair) --
taupre  = 20 * ms
taupost = 20 * ms

# -- Triplet STDP (excitatory; Pfister-Gerstner, on top of the pair rule) --
tau_x         = 100 * ms   # slow presynaptic detector r2
tau_y         = 125 * ms   # slow postsynaptic detector o2
A3PRE_CENTER  =  0.004     # triplet LTP amplitude (pre-post-post), > 0
A3POST_CENTER = -0.002     # triplet LTD amplitude (post-pre-pre), < 0

# -- Excitatory weight bounds --
wmin = 0.0

# -- Excitatory synapse --
WMAX_CENTER  = 1.0
APRE_CENTER  =  0.008
APOST_CENTER = -0.0096

# -- Inhibitory lateral synapse --
W_INH_CENTER = 1.0
W_INH_MIN    = 0.0
APRE_INH     = 0.004
APOST_INH    = -0.0048

# -- Channel layout --
N_CHANNELS    = 64
N_PER_CHANNEL = N_IN // N_CHANNELS   # 2

# -- Tonotopic plasticity (polynomial decay: max(0, 1-(d_channel/R)^p)) --
R_EXC_CHANNEL = 11
p_EXC         = 3

# -- Homeostatic normalisation (25ms tick, verbatim from train.py) --
NORM_LIMIT_EXC = 1.0455
NORM_LIMIT_INH = 0.40
NORM_DT        = 25 * ms

# -- Audio encoder kwargs (verbatim from train.py's call) --
SUST_GAIN  = 0.3
ONSET_GAIN = 3.0
ENCODER_KWARGS = dict(
    scale=1.0,
    num_filters=64,
    sustained_per_band=1,
    onset_per_band=1,
    phase_per_band=0,
    sust_gain=SUST_GAIN,
    onset_gain=ONSET_GAIN,
    sust_spread_min=1,
    sust_spread_max=1,
)

# -- Tensor transform layout --
OFFSETS = np.arange(-R_EXC_CHANNEL, R_EXC_CHANNEL + 1)   # -11..+11 (23 values)
N_OFF   = len(OFFSETS)

# Per-frame output shapes:
WEIGHTS_SHAPE  = (N_PER_CHANNEL, N_OFF, N_H)   # (2, 23, 128)
ACTIVITY_SHAPE = (N_IN,)                        # (128,)

# ═══════════════════════════════════════════════════════════════════════════════
# Read-only module-level precompute (deterministic, fork-safe) — verbatim formulas
# from train.py, ported to N_PER_CHANNEL=2/N_IN=N_H=128.
# ═══════════════════════════════════════════════════════════════════════════════

# -- Tonotopic excitatory matrices -------------------------------------------------
_ch_i    = (np.arange(N_IN) // N_PER_CHANNEL).reshape(-1, 1)
_ch_j    = (np.arange(N_H)  // N_PER_CHANNEL).reshape(1, -1)
_dist_ch = np.abs(_ch_i - _ch_j)
_dist_ch = np.minimum(_dist_ch, N_CHANNELS - _dist_ch)            # circular
_topo_exc = np.maximum(0.0, 1.0 - (_dist_ch / R_EXC_CHANNEL) ** p_EXC)
_mask_ih  = _dist_ch <= R_EXC_CHANNEL
_SRC_IH, _TGT_IH = np.where(_mask_ih)

_WMAX_MATRIX   = WMAX_CENTER   * _topo_exc
_APRE_MATRIX   = APRE_CENTER   * _topo_exc
_APOST_MATRIX  = APOST_CENTER  * _topo_exc
_A3PRE_MATRIX  = A3PRE_CENTER  * _topo_exc   # triplet LTP amplitude (topo-scaled)
_A3POST_MATRIX = A3POST_CENTER * _topo_exc   # triplet LTD amplitude (topo-scaled)
del _ch_i, _ch_j, _dist_ch, _topo_exc

# -- Jaccard inhibitory matrices ---------------------------------------------------
_ch_h       = np.arange(N_H) // N_PER_CHANNEL
_dist_ch_hh = np.abs(_ch_h.reshape(-1, 1) - _ch_h.reshape(1, -1))
_dist_ch_hh = np.minimum(_dist_ch_hh, N_CHANNELS - _dist_ch_hh)   # circular
_window_size = 2 * R_EXC_CHANNEL + 1
_overlap_ch  = np.maximum(0, _window_size - _dist_ch_hh)
_jaccard     = np.where(_overlap_ch > 0,
                        _overlap_ch / (_window_size + _dist_ch_hh), 0.0)
_mask_hh  = (_overlap_ch > 0) & (~np.eye(N_H, dtype=bool))
_SRC_HH, _TGT_HH = np.where(_mask_hh)

_WMAX_INH_MATRIX  = W_INH_CENTER * _jaccard
_APRE_INH_MATRIX  = APRE_INH     * _jaccard
_APOST_INH_MATRIX = APOST_INH    * _jaccard
del _ch_h, _dist_ch_hh, _overlap_ch, _jaccard

# -- Shared initial weight matrices (per-piece starting point) ---------------------
# Excitatory: formula-shaped, column-normalised to NORM_LIMIT_EXC (as in train.py init).
W_IH_INIT = np.zeros((N_IN, N_H))
W_IH_INIT[_SRC_IH, _TGT_IH] = _WMAX_MATRIX[_SRC_IH, _TGT_IH]
for _j in range(N_H):
    _rows = _SRC_IH[_TGT_IH == _j]
    _wsum = W_IH_INIT[_rows, _j].sum()
    if _wsum > 0:
        W_IH_INIT[_rows, _j] *= NORM_LIMIT_EXC / _wsum

# Inhibitory: uniform random init in [0.01, 0.02] on connected positions (train.py),
# drawn once from a fixed seed so every piece / wav / worker starts from the same state.
_rng = np.random.RandomState(42)
W_HH_INIT = np.zeros((N_H, N_H))
W_HH_INIT[_SRC_HH, _TGT_HH] = _rng.uniform(0.01, 0.02, size=_SRC_HH.shape[0])

# -- Transform index map: IN_IDX[t, o, j] = ((ch_j + offset) % 64) * 2 + t ---------
_ch_j_row = (np.arange(N_H) // N_PER_CHANNEL)                       # (N_H,)
_in_ch    = (_ch_j_row[None, :] + OFFSETS[:, None]) % N_CHANNELS    # (23, N_H)
IN_IDX = np.stack([_in_ch * N_PER_CHANNEL + t for t in range(N_PER_CHANNEL)])  # (2,23,N_H)
_J_ROW = np.broadcast_to(np.arange(N_H), WEIGHTS_SHAPE)             # (2,23,N_H)
del _ch_j_row, _in_ch


# ═══════════════════════════════════════════════════════════════════════════════
# Network construction  (call once per process — never rebuilt per piece/wav)
# ═══════════════════════════════════════════════════════════════════════════════

def build_network():
    """Build the Brian2 network fresh and return a handle namespace.

    Must be called once per worker process (Brian2 objects are not safe to fork
    already-built). The returned handle is passed to train_fingerprint_piecewise().
    """
    defaultclock.dt = DT_SIM

    # ── Input neurons ──────────────────────────────────────────────────────────
    eqs_in = """
    dv/dt = (-v - a) / tau_m + I_timed(t, i) / tau_current : 1
    da/dt = -a / tau_a : 1
    """
    G_in = NeuronGroup(N_IN, eqs_in, threshold="v > v_th_in",
                       reset="v=0; a+=beta", refractory=2 * ms, method="euler")
    G_in.namespace["I_timed"] = TimedArray(np.zeros((1, N_IN), dtype=float), dt=DT_SIM)

    # ── Hidden neurons ─────────────────────────────────────────────────────────
    eqs_h = f"""
    dv/dt       = -v / tau_h + sigma_noise * xi                       : 1
    dvth/dt     = -(vth - {vth_rest}) / tau_vth                       : 1
    dtrace_r/dt = -trace_r / tau_r                                    : 1
    """
    G_h = NeuronGroup(N_H, eqs_h, threshold="v > vth",
                      reset=f"v=0; vth=vth+{vth_jump}; trace_r=1;", method="euler")

    # ── Excitatory TRIPLET STDP synapses: input → hidden ───────────────────────
    stdp_model = """
    w          : 1
    dapre/dt   = -apre  / taupre  : 1 (event-driven)
    dapost/dt  = -apost / taupost : 1 (event-driven)
    dr1/dt     = -r1 / taupre      : 1 (event-driven)
    dr2/dt     = -r2 / tau_x       : 1 (event-driven)
    do1/dt     = -o1 / taupost     : 1 (event-driven)
    do2/dt     = -o2 / tau_y       : 1 (event-driven)
    wmax_syn   : 1
    Apre_syn   : 1
    Apost_syn  : 1
    A3pre_syn  : 1
    A3post_syn : 1
    """
    on_pre  = (f"v_post += w * (1 - trace_r_post)\n"
               f"apre += Apre_syn\n"
               f"w = clip(w + apost*(w-{wmin}) + A3post_syn*o1*r2*(w-{wmin}), {wmin}, wmax_syn)\n"
               f"r1 += 1\n"
               f"r2 += 1")
    on_post = (f"apost += Apost_syn\n"
               f"w = clip(w + apre*(wmax_syn-w) + A3pre_syn*r1*o2*(wmax_syn-w), {wmin}, wmax_syn)\n"
               f"o1 += 1\n"
               f"o2 += 1")

    S_ih = Synapses(G_in, G_h, model=stdp_model, on_pre=on_pre, on_post=on_post)
    S_ih.connect(i=_SRC_IH, j=_TGT_IH)
    src_ih = np.array(S_ih.i)
    tgt_ih = np.array(S_ih.j)
    S_ih.wmax_syn   = _WMAX_MATRIX[src_ih, tgt_ih]
    S_ih.Apre_syn   = _APRE_MATRIX[src_ih, tgt_ih]
    S_ih.Apost_syn  = _APOST_MATRIX[src_ih, tgt_ih]
    S_ih.A3pre_syn  = _A3PRE_MATRIX[src_ih, tgt_ih]
    S_ih.A3post_syn = _A3POST_MATRIX[src_ih, tgt_ih]
    # w defaults to 0 on Synapses creation — set it to the shared init BEFORE
    # net.store('init') below so restore('init') brings it back correctly every
    # piece without a separate manual reassignment (mirrors S_hh.w_inh below).
    S_ih.w          = W_IH_INIT[src_ih, tgt_ih]

    # ── Inhibitory lateral STDP synapses: hidden → hidden (soft-refractory gated) ─
    stdp_inh_model = """
    w_inh          : 1
    dapre_inh/dt   = -apre_inh  / taupre  : 1 (event-driven)
    dapost_inh/dt  = -apost_inh / taupost : 1 (event-driven)
    wmax_inh_syn   : 1
    Apre_inh_syn   : 1
    Apost_inh_syn  : 1
    """
    on_pre_inh  = (f"v_post -= w_inh * (1 - trace_r_post)\n"
                   f"apre_inh += Apre_inh_syn\n"
                   f"w_inh = clip(w_inh + apost_inh*(w_inh-{W_INH_MIN}), {W_INH_MIN}, wmax_inh_syn)")
    on_post_inh = (f"apost_inh += Apost_inh_syn\n"
                   f"w_inh = clip(w_inh + apre_inh*(wmax_inh_syn-w_inh), {W_INH_MIN}, wmax_inh_syn)")

    S_hh = Synapses(G_h, G_h, model=stdp_inh_model, on_pre=on_pre_inh, on_post=on_post_inh)
    S_hh.connect(i=_SRC_HH, j=_TGT_HH)
    src_hh = np.array(S_hh.i)
    tgt_hh = np.array(S_hh.j)
    S_hh.wmax_inh_syn  = _WMAX_INH_MATRIX[src_hh, tgt_hh]
    S_hh.Apre_inh_syn  = _APRE_INH_MATRIX[src_hh, tgt_hh]
    S_hh.Apost_inh_syn = _APOST_INH_MATRIX[src_hh, tgt_hh]
    S_hh.w_inh         = W_HH_INIT[src_hh, tgt_hh]

    # ── Count-only spike monitors (per-neuron cumulative firing, never reset
    #    within a piece — used to diff per-repeat spike counts) ────────────────
    spike_in  = SpikeMonitor(G_in, record=False)
    spike_hid = SpikeMonitor(G_h,  record=False)

    # ── Vectorised L1 normalisation every 25 ms (verbatim tick from train.py) ──
    wmax_syn_arr     = np.array(S_ih.wmax_syn)
    wmax_inh_syn_arr = np.array(S_hh.wmax_inh_syn)

    @network_operation(dt=NORM_DT, when='end', order=0)
    def normalize_weights():
        # Excitatory: scale any column whose L1 exceeds NORM_LIMIT_EXC.
        w = np.array(S_ih.w)
        col_sum = np.bincount(tgt_ih, weights=w, minlength=N_H)
        scale = np.where(col_sum > NORM_LIMIT_EXC, NORM_LIMIT_EXC / col_sum, 1.0)
        S_ih.w[:] = np.clip(w * scale[tgt_ih], wmin, wmax_syn_arr)

        # Inhibitory: same, against NORM_LIMIT_INH.
        wi = np.array(S_hh.w_inh)
        col_sum_i = np.bincount(tgt_hh, weights=wi, minlength=N_H)
        scale_i = np.where(col_sum_i > NORM_LIMIT_INH, NORM_LIMIT_INH / col_sum_i, 1.0)
        S_hh.w_inh[:] = np.clip(wi * scale_i[tgt_hh], W_INH_MIN, wmax_inh_syn_arr)

    # ── In-run repeat-boundary snapshot capture ─────────────────────────────────
    # Piece-wise training tiles N_REPEATS copies of one piece's input into a
    # single TimedArray and used to run N_REPEATS separate net.run(WINDOW_MS)
    # calls just to pause and read out per-repeat weight/count snapshots between
    # them (no restore between repeats — see train_fingerprint_piecewise). Each
    # net.run() call carries real fixed overhead in Brian2 (schedule checks,
    # before_run() hooks on every object) even under cython codegen, so N_REPEATS
    # calls per piece x ~48 pieces/wav = hundreds of calls purely for bookkeeping.
    #
    # This network_operation instead runs *within* a single continuous
    # net.run(WINDOW_MS*N_REPEATS) call and does the same bookkeeping itself.
    # IMPORTANT: it deliberately has NO dt/clock argument, so it shares
    # defaultclock exactly (same Clock object as G_in/G_h/S_ih/S_hh) rather than
    # getting its own separate dt=WINDOW_MS Clock. An earlier version used a
    # separate dt=WINDOW_MS clock and every tick's captured state turned out to
    # be exactly one defaultclock step (DT_SIM) later than the nominal boundary
    # (verified empirically — a Brian2 multi-clock synchronization quirk, not
    # documented behaviour to rely on). Sharing defaultclock and doing the
    # "once every WINDOW_MS steps" check with a plain Python counter instead
    # sidesteps that quirk entirely and was verified bit-for-bit equivalent to
    # the old per-rep net.run() loop across several parameter combinations
    # (see scratchpad/equivalence_sweep.py). It fires every simulated
    # millisecond, but only reads/copies state on the 1-in-WINDOW_MS steps where
    # the counter condition holds — the other calls are a single integer
    # increment + modulo check, negligible next to a net.run() call's overhead.
    # order=1 (> normalize_weights' order=0) guarantees normalization always
    # runs first on steps the two share (every WINDOW_MS, since WINDOW_MS is a
    # multiple of NORM_DT), matching the original call-boundary ordering where
    # normalization always preceded the Python-level snapshot readout.
    _win_steps = int(round(WINDOW_MS * ms / DT_SIM))
    _piece_state = {"step": 0, "w_snapshots": [], "in_cum": [], "hid_cum": []}

    @network_operation(when='end', order=1)
    def capture_repeat_boundary():
        _piece_state["step"] += 1
        if _piece_state["step"] % _win_steps == 0:
            _piece_state["w_snapshots"].append(np.array(S_ih.w))
            _piece_state["in_cum"].append(np.array(spike_in.count, dtype=np.int64))
            _piece_state["hid_cum"].append(np.array(spike_hid.count, dtype=np.int64))

    net = Network(G_in, G_h, S_ih, S_hh, spike_in, spike_hid,
                  normalize_weights, capture_repeat_boundary)
    G_h.vth = vth_init
    net.store('init')   # clean snapshot: clock=0, v=0, a=0, vth=vth_init, w/w_inh at
                         # shared init, apre/apost/r1/r2/o1/o2/apre_inh/apost_inh=0,
                         # monitors empty

    return SimpleNamespace(
        net=net, G_in=G_in, G_h=G_h, S_ih=S_ih, S_hh=S_hh,
        src_ih=src_ih, tgt_ih=tgt_ih, src_hh=src_hh, tgt_hh=tgt_hh,
        spike_in=spike_in, spike_hid=spike_hid, piece_state=_piece_state,
    )


# ═══════════════════════════════════════════════════════════════════════════════
# Per-wav, piece-wise training → sequence of frames
# ═══════════════════════════════════════════════════════════════════════════════

def train_fingerprint_piecewise(h, wav_path):
    """Run train.py's exact piece-wise scheme over one wav; return
    (weights_frames, ia_frames, ha_frames) or None if the wav can't be encoded /
    is shorter than one window.

    weights_frames : (N_PIECES, 2, 23, 128) float16
    ia_frames       : (N_PIECES, 128)        float16
    ha_frames       : (N_PIECES, 128)        float16

    Per piece: net.restore('init') once (shared init weights, matching train.py —
    never carried over from the previous piece), then ONE net.run() covering all
    N_REPEATS repeats (no restore between them — state carries over continuously,
    exactly as train.py). The per-repeat weight/count snapshots that used to
    require N_REPEATS separate net.run() calls are instead captured in-run by
    h.piece_state (populated by the capture_repeat_boundary network_operation
    registered in build_network — see its docstring for why this is safe and
    equivalent). After the run, snapshots for repeats >= SNAPSHOT_FROM_REPEAT are
    averaged (weights) / summed (counts) exactly as before.
    """
    try:
        I, T_sim = compute_spike_input_current(wav_path, **ENCODER_KWARGS)
    except Exception as e:
        print(f"  [skip {wav_path}: encode error: {e}]")
        return None

    n_pieces = (T_sim - WINDOW_MS) // HOP_MS + 1
    if n_pieces < 1:
        print(f"  [skip {wav_path}: clip ({T_sim}ms) shorter than one window ({WINDOW_MS}ms)]")
        return None

    weights_frames = np.empty((n_pieces,) + WEIGHTS_SHAPE, dtype=np.float16)
    ia_frames      = np.empty((n_pieces, N_IN), dtype=np.float16)
    ha_frames      = np.empty((n_pieces, N_H),  dtype=np.float16)

    for piece_idx in range(n_pieces):
        t_start = piece_idx * HOP_MS
        t_end   = t_start + WINDOW_MS
        I_piece = I[:, t_start:t_end]                 # (N_IN, WINDOW_MS)
        I_tiled = np.tile(I_piece, (1, N_REPEATS))     # (N_IN, WINDOW_MS * N_REPEATS)

        # ── Reset to the shared clean init for this piece (never carried over
        #    from the previous piece — matches train.py). build_network() sets
        #    S_ih.w and S_hh.w_inh to their shared-init values BEFORE
        #    net.store('init') is called, so restore('init') alone already resets
        #    w, w_inh, apre(_inh), apost(_inh), r1, r2, o1, o2 to the correct
        #    per-piece starting state — no manual reassignment needed here.
        h.net.restore('init')
        h.G_in.namespace["I_timed"] = TimedArray(I_tiled.T.astype(float), dt=DT_SIM)

        # h.piece_state's contents are plain Python objects, not Brian2 state, so
        # restore('init') above does not reset them — do it explicitly, in place
        # (the capture_repeat_boundary closure holds a reference to this exact
        # dict, so it must be mutated, not replaced).
        h.piece_state["step"] = 0
        h.piece_state["w_snapshots"].clear()
        h.piece_state["in_cum"].clear()
        h.piece_state["hid_cum"].clear()

        # ── Simulate all N_REPEATS repeats in ONE call. No restore between
        #    repeats — state carries over continuously, giving genuine repeated
        #    exposure rather than N independent trials (same as before). The
        #    capture_repeat_boundary network_operation (registered once in
        #    build_network) appends one snapshot per repeat boundary during
        #    this single run.
        h.net.run(WINDOW_MS * N_REPEATS * DT_SIM)

        w_hist  = np.stack(h.piece_state["w_snapshots"])   # (N_REPEATS, n_synapses)
        in_cum  = np.stack(h.piece_state["in_cum"])         # (N_REPEATS, N_IN)
        hid_cum = np.stack(h.piece_state["hid_cum"])        # (N_REPEATS, N_H)
        if w_hist.shape[0] != N_REPEATS:
            raise RuntimeError(
                f"expected {N_REPEATS} repeat-boundary snapshots from "
                f"capture_repeat_boundary, got {w_hist.shape[0]} — the per-timestep "
                "counter logic in build_network() may be wrong for this Brian2 "
                f"version (piece {piece_idx}, wav {wav_path})"
            )

        # w_hist[k] / in_cum[k] / hid_cum[k] (k=0..N_REPEATS-1) is the state
        # right AFTER repeat k completes -- capture_repeat_boundary's own
        # counter fires exactly on those steps (own-clock design, verified
        # bit-for-bit equivalent to the old per-rep loop; see build_network()
        # and scratchpad/equivalence_sweep.py). Average the weight snapshots
        # for repeats SNAPSHOT_FROM_REPEAT..N_REPEATS-1; sum the spike counts
        # accumulated over that same window (cumulative counts, so it's a
        # single subtraction against the count at the end of the repeat just
        # before the window, or zero if the window starts at repeat 0).
        w_mean_sparse = w_hist[SNAPSHOT_FROM_REPEAT:].mean(axis=0)   # (n_synapses,)
        piece_weight_avg = np.zeros((N_IN, N_H))
        piece_weight_avg[h.src_ih, h.tgt_ih] = w_mean_sparse

        if SNAPSHOT_FROM_REPEAT == 0:
            base_in  = np.zeros(N_IN, dtype=np.int64)
            base_hid = np.zeros(N_H,  dtype=np.int64)
        else:
            base_in  = in_cum[SNAPSHOT_FROM_REPEAT - 1]
            base_hid = hid_cum[SNAPSHOT_FROM_REPEAT - 1]
        in_counts_sum  = in_cum[-1]  - base_in
        hid_counts_sum = hid_cum[-1] - base_hid

        # ── Extract updated weights ────────────────────────────────────────────
        w_frame, ia_frame, ha_frame = fingerprint_to_sample_piece(
            piece_weight_avg, in_counts_sum, hid_counts_sum
        )
        weights_frames[piece_idx] = w_frame
        ia_frames[piece_idx]      = ia_frame
        ha_frames[piece_idx]      = ha_frame

    return weights_frames, ia_frames, ha_frames


# ═══════════════════════════════════════════════════════════════════════════════
# Tensor transform → one frame's ANN-ready output
# ═══════════════════════════════════════════════════════════════════════════════

def _normalize_activity(counts):
    """Per-frame [0,1] via ÷99th-percentile + clip. Preserves 0 (silent)."""
    denom = np.percentile(counts, 99)
    if denom <= 0:
        return np.zeros_like(counts, dtype=np.float16)
    return np.clip(counts / denom, 0.0, 1.0).astype(np.float16)


def fingerprint_to_sample_piece(fp, in_counts, hid_counts):
    """(N_IN,N_H) mean weight matrix + summed spike counts (over the last
    N_REPEATS-SNAPSHOT_FROM_REPEAT repeats of one piece) -> one frame's output.

    weights         (2,23,128) float16 — raw type-images, silent cells zeroed
    input_activity  (128,)     float16 — per-frame [0,1]
    hidden_activity (128,)     float16 — per-frame [0,1]
    """
    weights = fp[IN_IDX, _J_ROW].astype(np.float32)            # (2,23,N_H)

    # Zero any weight whose presynaptic input neuron OR hidden neuron never fired
    # (over the same repeats-4..7 window the weights were averaged over).
    in_silent  = (in_counts == 0)
    hid_silent = (hid_counts == 0)
    silent = in_silent[IN_IDX] | hid_silent[None, None, :]      # (2,23,N_H)
    weights[silent] = 0.0

    return (weights.astype(np.float16),
            _normalize_activity(in_counts),
            _normalize_activity(hid_counts))


# ═══════════════════════════════════════════════════════════════════════════════
# Multiprocessing worker helpers (imported by spawn workers from this real module)
# ═══════════════════════════════════════════════════════════════════════════════

_H = None   # per-process Brian2 network handle


def _init_worker(log_every=50):
    """Pool initializer: build one Brian2 network per worker process."""
    global _H
    _H = build_network()


def process_one(entry):
    """Generate one wav's fingerprint MOVIE. `entry` is a plain dict (picklable
    for spawn). Always returns (person_id, record_id, label, payload) so the
    driver can track per-person completion even for skipped wavs. `payload` is
    (weights_frames, ia_frames, ha_frames) on success, or None if the wav could
    not be encoded / was too short.
    """
    pid, rid, lab = entry['person_id'], entry['record_id'], entry['label']
    out = train_fingerprint_piecewise(_H, entry['wav_path'])
    if out is None:
        return (pid, rid, lab, None)
    return (pid, rid, lab, out)


In [ ]:
# ── Enumerate wavs: every person, every session, first PER_SESSION wavs (sorted) ──
from pathlib import Path

root = Path(INPUT_ROOT)
assert root.is_dir(), f"INPUT_ROOT not found: {root}"

entries = []
for person_dir in sorted(d for d in root.iterdir() if d.is_dir()):
    for sess_dir in sorted(d for d in person_dir.iterdir() if d.is_dir()):
        wavs = sorted(f.name for f in sess_dir.iterdir()
                      if f.is_file() and f.suffix.lower() in AUDIO_EXTS)[:PER_SESSION]
        for wav in wavs:
            entries.append(dict(
                person_id=person_dir.name,
                record_id=sess_dir.name,
                wav=wav,
                label=f"{person_dir.name}/{sess_dir.name}/{wav}",
                wav_path=str(sess_dir / wav),
            ))

n_persons  = len({e["person_id"] for e in entries})
n_sessions = len({(e["person_id"], e["record_id"]) for e in entries})
print(f"{len(entries)} wavs  |  {n_persons} persons  |  {n_sessions} sessions")
assert entries, "no audio found under INPUT_ROOT — check the path / AUDIO_EXTS"

In [ ]:
# ── Generate fingerprint MOVIES in parallel (spawn), then write ONE shard DIRECTORY ──
# Logs one line each time a person's wavs are all done (successful + skipped).
# Each wav now yields a variable-length SEQUENCE of frames (one per piece), so
# results are stored CSR-style: all frames concatenated + a per-sample n_pieces
# count (offsets = cumsum([0]+n_pieces)) instead of one fixed-shape array per wav.
#
# The big frame arrays are saved as PLAIN .npy files (not bundled into a .npz) so
# ann_backend_sequence.ipynb can open them with mmap_mode='r' for true lazy loading
# -- an .npz is a zip archive, and numpy does NOT actually lazy-load zip members
# even when mmap_mode is passed (verified: opening a 500MB array from an .npz with
# mmap_mode='r' pulls the full 500MB into RSS immediately; a plain .npy does not).
# At full-corpus scale (~95GB across both splits) that distinction is the difference
# between fitting in Kaggle RAM and OOMing.
import multiprocessing as mp
import time
from collections import defaultdict
import numpy as np
import _fingerprint_core_piecewise as C

# Expected wavs per person → lets us detect when a person is fully processed even
# though imap_unordered returns results in arbitrary order.
expected = defaultdict(int)
for e in entries:
    expected[e["person_id"]] += 1
n_persons_total = len(expected)

t0 = time.time()
print("warming Brian2 codegen cache ...", flush=True)
C.build_network()                       # compile once in the parent; workers reuse the cache

results = []                            # (weights_frames, ia_frames, ha_frames, pid, rid, label)
processed = defaultdict(int)
persons_done = n_wavs_done = n_skipped = 0

ctx = mp.get_context("spawn")
with ctx.Pool(WORKERS, initializer=C._init_worker, initargs=(50,)) as pool:
    for pid, rid, lab, payload in pool.imap_unordered(C.process_one, entries):
        n_wavs_done += 1
        processed[pid] += 1
        if payload is not None:         # None = unencodable / too-short wav (skipped)
            w, ia, ha = payload
            results.append((w, ia, ha, pid, rid, lab))
        else:
            n_skipped += 1
        if processed[pid] == expected[pid]:
            persons_done += 1
            print(f"[person {persons_done}/{n_persons_total}] {pid} done  |  "
                  f"{len(results)} fingerprints, {n_skipped} skipped, "
                  f"{n_wavs_done}/{len(entries)} wavs  |  {time.time()-t0:.0f}s",
                  flush=True)

assert results, "no fingerprints produced"
print(f"produced {len(results)} fingerprint movies ({n_skipped} skipped) "
      f"over {n_persons_total} persons in {time.time()-t0:.0f}s")

# ── Stack CSR-style: concatenate every sample's frames along axis 0, track
#    per-sample frame counts (n_pieces) instead of a fixed leading dim ─────────
n_pieces = np.array([r[0].shape[0] for r in results], dtype=np.int32)

weights_frames         = np.concatenate([r[0] for r in results], axis=0).astype(np.float16)
input_activity_frames  = np.concatenate([r[1] for r in results], axis=0).astype(np.float16)
hidden_activity_frames = np.concatenate([r[2] for r in results], axis=0).astype(np.float16)
person_ids = np.array([r[3] for r in results])
record_ids = np.array([r[4] for r in results])
labels     = np.array([r[5] for r in results])

print(f"weights_frames.shape         = {weights_frames.shape}  (TOTAL_FRAMES, 2, 23, 128)")
print(f"input_activity_frames.shape  = {input_activity_frames.shape}")
print(f"hidden_activity_frames.shape = {hidden_activity_frames.shape}")
print(f"n_pieces: min={n_pieces.min()} max={n_pieces.max()} mean={n_pieces.mean():.1f} "
      f"total={n_pieces.sum()} (== TOTAL_FRAMES: {n_pieces.sum() == weights_frames.shape[0]})")

# ── Write the shard as a DIRECTORY: 3 plain .npy files (mmap-able) + one tiny
#    meta.npz (person/session ids, labels, n_pieces, provenance -- always loaded
#    eagerly, negligible size regardless) ────────────────────────────────────────
shard_dir = os.path.join(WORK_DIR, OUTPUT_NAME)
os.makedirs(shard_dir, exist_ok=True)

np.save(os.path.join(shard_dir, "weights_frames.npy"), weights_frames)
np.save(os.path.join(shard_dir, "input_activity_frames.npy"), input_activity_frames)
np.save(os.path.join(shard_dir, "hidden_activity_frames.npy"), hidden_activity_frames)
np.savez(os.path.join(shard_dir, "meta.npz"),
         n_pieces=n_pieces,
         person_ids=person_ids, record_ids=record_ids, labels=labels,
         piece_window_ms=np.int32(C.WINDOW_MS),
         piece_hop_ms=np.int32(C.HOP_MS),
         n_repeats=np.int32(C.N_REPEATS),
         snapshot_from_repeat=np.int32(C.SNAPSHOT_FROM_REPEAT))

shard_size_mb = sum(
    os.path.getsize(os.path.join(shard_dir, f))
    for f in os.listdir(shard_dir)
) / 1e6
print(f"saved shard → {shard_dir}/  ({shard_size_mb:.1f} MB: weights_frames.npy, "
      f"input_activity_frames.npy, hidden_activity_frames.npy, meta.npz)")
print("Zip this directory before uploading as a Kaggle dataset input, e.g.:\n"
      f"  !cd {WORK_DIR} && zip -r -0 {OUTPUT_NAME}.zip {OUTPUT_NAME}/\n"
      "(-0 = store, no compression -- Kaggle unzips datasets on mount, and the .npy "
      "files inside must stay uncompressed on disk for mmap to work after unzip; "
      "zip's own compression only affects the upload transfer, not the mounted files, "
      "but -0 keeps this predictable and skips wasted CPU on already-tiny fp16 data.)")


In [ ]:
# ── Verify the written shard (directory of .npy + meta.npz) ───────────────────
import numpy as np
W  = np.load(os.path.join(shard_dir, "weights_frames.npy"),         mmap_mode="r")
IA = np.load(os.path.join(shard_dir, "input_activity_frames.npy"),  mmap_mode="r")
HA = np.load(os.path.join(shard_dir, "hidden_activity_frames.npy"), mmap_mode="r")
meta = np.load(os.path.join(shard_dir, "meta.npz"), allow_pickle=True)

for name, a in (("weights_frames", W), ("input_activity_frames", IA), ("hidden_activity_frames", HA)):
    a_small = np.asarray(a[:1000])   # only touch a small slice -- don't force-materialize the shard
    print(f"  {name:24s} {str(a.shape):24s} {a.dtype}  "
          f"sample range [{float(a_small.min()):.4f}, {float(a_small.max()):.4f}]")

n_pieces = meta["n_pieces"]
print("  samples  :", len(meta["labels"]), "== len(n_pieces):", len(n_pieces) == len(meta["labels"]))
print("  persons  :", len(set(meta["person_ids"].tolist())))
print("  sessions :", len({(p, r) for p, r in zip(meta["person_ids"], meta["record_ids"])}))
print(f"  n_pieces : min={n_pieces.min()} max={n_pieces.max()} mean={n_pieces.mean():.1f}")
print(f"  piece_window_ms={int(meta['piece_window_ms'])}  piece_hop_ms={int(meta['piece_hop_ms'])}  "
      f"n_repeats={int(meta['n_repeats'])}  snapshot_from_repeat={int(meta['snapshot_from_repeat'])}")

# CSR offset round-trip: offsets[i]:offsets[i+1] must slice sample i's frames,
# and the last offset must equal TOTAL_FRAMES.
offsets = np.concatenate([[0], np.cumsum(n_pieces)])
assert offsets[-1] == W.shape[0], "CSR offsets don't cover all frames"

# spot-check one sample's slice without materializing the whole shard
i = 0
sl = np.asarray(W[offsets[i]:offsets[i + 1]])
assert sl.shape == (n_pieces[i], 2, 23, 128)
print(f"  spot-check: sample 0 slice shape {sl.shape} matches n_pieces[0]={n_pieces[0]}")

ia_small = np.asarray(IA[:1000]); ha_small = np.asarray(HA[:1000])
assert 0.0 <= float(ia_small.min()) and float(ia_small.max()) <= 1.0
assert 0.0 <= float(ha_small.min()) and float(ha_small.max()) <= 1.0
print("OK — activities in [0,1] (sampled), CSR offsets consistent; zip this shard directory "
      "and upload as a Kaggle dataset.")
